# CSCI443 HW 6 EDA, Data Cleaning, and Hypothesis Testing.

In this homework we explore some of the capabilities of Apache Spark as related
by creating a reporting pipeline for Olist.

We will be using 

  https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

This dataset spans 100k orders from 2016 and 2018.  We will build a pipeline
for running periodic hypothesis tests to detect anomalies.



In [0]:
from pyspark.sql import SparkSession

# create a spark session.
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()


## Part I: Multiple Choice Regarding Recent Lectures

**1.1. What is an AWS Region?**  
A. A virtual machine  
B. A geographic area containing multiple availability zones  
C. A storage bucket  
D. A single data center  
**Answer: B**

**1.2. What is an Availability Zone (AZ)?**  
A. A backup snapshot  
B. A logical partition in S3  
C. An isolated data center within a region  
D. A collection of regions  
**Answer: C**

**1.3. Why use multiple AZs?**  
A. To compress data  
B. To increase CPU speed  
C. For fault tolerance and high availability  
D. To reduce storage cost  
**Answer: C**

**1.4. What is S3 best described as?**  
A. In-memory cache  
B. Object (blob) storage  
C. Block storage  
D. File system  
**Answer: B**

**1.5. What is a key property of S3?**  
A. Direct memory access  
B. Low latency disk access  
C. Highly durable, scalable storage  
D. Strong consistency for compute  
**Answer: C**

**1.6. EBS is best described as:**  
A. Block storage attached to instances  
B. Distributed file system  
C. Object storage  
D. Cache layer  
**Answer: A**

**1.7. Instance Store is:**  
A. Persistent storage across reboots  
B. Object storage  
C. Temporary storage tied to instance lifecycle  
D. Network storage  
**Answer: C**

**1.8. Key difference between EBS and Instance Store:**  
A. Instance store is persistent  
B. EBS persists after instance stops; instance store does not  
C. Instance store is object storage  
D. EBS is slower CPU  
**Answer: B**

**1.9. When should you use Instance Store?**  
A. Temporary scratch space  
B. Critical databases  
C. Backups  
D. Long-term storage  
**Answer: A**

**1.10. When should you use S3?**  
A. CPU processing  
B. Long-term scalable storage  
C. Fast local disk access  
D. In-memory caching  
**Answer: B**

**1.11. The Spark driver is responsible for:**  
A. Executing all tasks  
B. Planning and coordinating execution  
C. Networking only  
D. Storing all data  
**Answer: B**

**1.12. Executors are responsible for:**  
A. Managing UI  
B. Running tasks on partitions  
C. Scheduling jobs  
D. Query planning  
**Answer: B**

**1.13. Narrow transformations:**  
A. Always return scalars  
B. Do not require data movement between partitions  
C. Always aggregate  
D. Require shuffle  
**Answer: B**

**1.14. Wide transformations:**  
A. Are always faster  
B. Require data movement across partitions  
C. Never shuffle  
D. Only apply to small data  
**Answer: B**

**1.15. Example of narrow transformation:**  
A. reduceByKey  
B. filter  
C. join  
D. groupBy  
**Answer: B**

**1.16. Example of wide transformation:**  
A. select  
B. filter  
C. map  
D. groupBy  
**Answer: D**

**1.17. Actions in Spark:**  
A. Trigger execution  
B. Build logical plans  
C. Are lazy  
D. Always return DataFrames  
**Answer: A**

**1.18. Transformations in Spark:**  
A. Write to disk  
B. Execute immediately  
C. Return scalars  
D. Are lazily evaluated  
**Answer: D**

**1.19. Inner join returns:**  
A. All rows from left  
B. All rows with nulls  
C. Only matching rows  
D. All rows from right  
**Answer: C**

**1.20. Left join returns:**  
A. All rows from left + matches  
B. Only unmatched rows  
C. All rows from right  
D. Only matching rows  
**Answer: A**

**1.21. What does `groupBy` do in Spark?**  
A. Sorts rows by a column  
B. Groups rows with the same key for aggregation  
C. Filters rows based on a condition  
D. Joins two DataFrames  
**Answer: B**

**1.22. What does `orderBy` do in Spark?**  
A. Groups rows for aggregation  
B. Filters rows  
C. Sorts rows based on one or more columns  
D. Removes duplicates  
**Answer: C**

**1.23. ETL stands for:**  
A. Extract Transform Load  
B. Extract Transfer Load  
C. Execute Transform Load  
D. Evaluate Transform Load  
**Answer: A**

**1.24. ELT differs from ETL because:**  
A. It loads data before transformation  
B. It skips transformation  
C. It transforms before extraction  
D. It uses no storage  
**Answer: A**

**1.25. A data lake is:**  
A. SQL-only system  
B. Raw storage for all data types  
C. In-memory cache  
D. Structured database only  
**Answer: B**

**1.26. Data warehouse is best for:**  
A. Temporary storage  
B. Structured analytics and BI  
C. Unstructured logs  
D. Real-time logging  
**Answer: B**

**1.27. Data lakehouse combines:**  
A. Data lake + warehouse features  
B. Cache + compute  
C. ETL + ELT  
D. CPU + memory  
**Answer: A**

**1.28. Why use a data lake for ML?**  
A. Smaller storage  
B. Faster CPU  
C. Stores raw and diverse data  
D. Only structured data  
**Answer: C**

**1.29. When use data warehouse?**  
A. Temporary files  
B. GPU compute  
C. BI and reporting  
D. Unstructured logs  
**Answer: C**

**1.30. When use data lakehouse?**  
A. Only raw data  
B. Unified analytics and ML  
C. No storage  
D. Only SQL queries  
**Answer: B**

**1.31. CSV advantage:**  
A. Columnar  
B. Schema enforced  
C. Human readable  
D. Compact binary  
**Answer: C**

(C is arguably true.  The others are just wrong.)

**1.32. CSV disadvantage:**  
A. Widely supported  
B. No schema enforcement  
C. Easy to parse  
D. Human readable  
**Answer: B**

**1.33. JSON advantage:**  
A. Flat structure  
B. Supports nested data  
C. Columnar  
D. Smaller than Parquet  
**Answer: B**

**1.34. JSON disadvantage:**  
A. Binary format  
B. Verbose and large  
C. Requires schema  
D. Cannot store arrays  
**Answer: B**

**1.35. Parquet advantage:**  
A. No schema  
B. Human readable  
C. Columnar and compressed  
D. Row-based  
**Answer: C**

**1.36. Why columnar layout?**  
A. Easier parsing  
B. Efficient reads of selected columns  
C. No compression  
D. Faster writes only  
**Answer: B**

**1.37. Delta Lake adds:**  
A. In-memory compute  
B. No schema  
C. ACID transactions to Parquet  
D. Row-based storage  
**Answer: C**

**1.38. ACID stands for:**  
A. Atomicity, Compression, Isolation, Data  
B. Accuracy, Control, Isolation, Data  
C. Atomicity, Consistency, Isolation, Durability  
D. None  
**Answer: C**

**1.39. Example of atomicity:**  
A. Data compression  
B. All-or-nothing transaction  
C. Partial write allowed  
D. Parallel execution  
**Answer: B**

**1.40. Time travel refers to:**  
A. Faster queries  
B. Backup only  
C. Accessing previous versions of data  
D. Data compression  
**Answer: C**

## Part II: Upload the Olist Data

**Problem 2.1.** Upload the data and ouput a list of the files here.

I uploaded the Olist data to



In [0]:
import os

# DS refers to Data Set.
DS = "/Volumes/workspace/default/files/olist"
files = os.listdir(DS)

for f in files:
    print(f)

Locally the files are at

In [0]:
import pandas as pd
from glob import glob
import os

DS = "archive"
csv_files = glob(os.path.join(DS, "*.csv"))
display(sorted(csv_files))


**Problem 2.2.** Load `olist_customers_dataset.csv` into a Spark DataFrame and output the first ten records using display to pretty-print the DataFrame.

   This confirms that we are able to access the data from within Databricks.


In [0]:
DS = "/Volumes/workspace/default/files/olist"

df = spark.read.csv(
    os.path.join(DS, "olist_customers_dataset.csv"),
    header=True,
    inferSchema=True
)
display(df.limit(10))

## Exploratory Data Analysis

The point of exploratory data analysis is to make sense of the data.

Exploratory data analysis encompasses activies like

1. Understanding the data structure
   *  look at the number of rows and columns to determine the scale of the data.
   *  look column labels, value types, and value ranges.
   *  identify categorical data and values taken by each kind of categorical data
   *  summarize the semantics of each column.

2. Summarizing data
   * computing summary statistics like means, stddevs, and counts.
     
3. Identifying missing values.
   * counting number of missing values and for which fields.
   * This can be used later in determining what kind of data cleaning may be needed.

4. Visualizing the data
5. Detecting outliers
   * create box plots of each type of data. Using 1.5 IQR rule of thumb to
     identify outliers.

6. Assessing relationships
   * create correlation matrices
   * create contingency tables for categorical data
   * perform preliminary hypothesis tests.
     

### Part III.A. Customer Data

%md

**Problem 3.A.1.** List the size of each data file.  What is the physical memory in your local machine?  Is the aggregate size of the data files larger than the physical memory on your local machine?
If the aggregate size of the datasets is larger than your physical memory then this is a good indicator that Pandas is not the right tool.  Pandas typically reads entire DataFrames into physical memory. 

Note: We will use Databricks regardless of the answer to this question, but it is generally
a good first step.  If the data is small, Pandas is often the correct answer.  Most modern laptops have
8GB or more of physical memory.  16GB or 3GB is common.   If the dataset size is >10 GB, Spark on a cluster
may be more appropriate.

**Answer 3.A.1** 

In [0]:
import os

DS = "/Volumes/workspace/default/files/olist"
files = [f for f in os.listdir(DS) if os.path.isfile(os.path.join(DS, f))]

file_sizes = [(f, os.path.getsize(os.path.join(DS, f))) for f in files]
total_size = sum(size for _, size in file_sizes)

df_sizes = spark.createDataFrame(file_sizes, ["filename", "size_bytes"])
display(df_sizes)
print(f"Total size: {total_size} bytes")

The total size of the files is around 120 MB.  My MacBook Pro has 32GB of memory which is more than sufficient to hold all of this data in memory at the same time.  Pandas operating on local data would do fine in this case.

This is not big enough to justify using Spark, but for educational purposes we will use Spark anyway.   

**Problem 3.A.2.** Let's perform some exploratory data analysis of `olist_customers_dataset.csv`.
We already loaded it using `read_csv` and we looked at the first few records.
List the fields and write one sentence describing what each means.  Read about the
data online to gain some understanding.

**customer_id** is a unique ID for a specific order for a customer.  There is one such ID per transaction.

**customer_unique_id** is a unique ID for a person.  The ID may appear in multiple transactions.

**customer_zip_code_prefix** is the first five digits of the postal zip code for this purchase transaction.

**customer_city** is the city in the customer address for this purchase transaction.

**customer_state** is the state in the customer address for this purchase transaction.

Addresses are associated with each order because a customer's location can change.

**Problem 3.A.3.** What is the difference between `customer_id` and `customer_unique_id`?  Which would we expect to have more unique IDs?  

`customer_id` is a unique ID for a specific transaction. 

`customer_unique_id` is a unique ID for a person. 

Because there is a different customer ID for each transaction, and I would expect there to be some customers that made more than one order so the number of `customer_unique_id` should be less than or equal tot he number of `customer_id`.

**Problem 3.A.4.** How many unique `customer_ids` are there?  How many unique `customer_unique_ids`? 

In [0]:
print("unique customer_ids", df.select("customer_id").distinct().count())
print("unique customer_unique_ids", df.select("customer_unique_id").distinct().count())

**Problem 3.A.5.** Can a `customer_unique_id` reasonably appear more than once in the file?  If so, under what conditions?

**Answer 3.A.5:** The choice of names for `customer_unique_id` and `customer_id`
feels a little misleading, but the definitions and usage are clear once one
studies the schema and confirms the interpretation by looking at the data itself.
Based on the name alone, one migth think that `customer_unique_id` cannot appear
more than once in the file; otherwise, it would not be unique, but that would an
incorrect interpretation.   The `customer_unique_id` is unique per customer while
the data has a row per customer transaction.  A single customer may make more
than one order so any `customer_unique_id` may appear more than once.

**Problem 3.A.6.** Write code to determine  if any fields are missing data in  `olist_customers_dataset.csv`.  Use isnull() as a first pass.
For numeric data, check for zero.  Are there any fields where zero is reasonable?
For string categorical data, check for empty strings, "NA", "n/a", and "None" using case-insensitive comparisons.  Did you find any rows suggesting data has been omitted?   If so, how many rows? 

Whenever performing Exploratory Data Analysis (EDA) we should look to see if there
is any missing data.  If so, we either omit those rows or we use *imputation* to 
infer appropriate values for the missing data.  Either option risks introducing
bias, so we must be careful to justify our decisions.  For this problem however, I am
only asking for you to identify whether there is missing data and if yes then how 
many rows are affected in the DataFrame.

In [0]:
df_nulls = df.select([df[col].isNull().alias(col) for col in df.columns])

# isNull() creates a boolean for each field that is true iff the field isNull().
# cast("int") converts the boolean for each field into a 1 for true or a 0
# for false.  We then sum the number of fields (columns) that evaluted to isNull()
# in a given row.   If the number that evaluated as isNull() is non-zero then the
# sum is greater than 0.  df.filter selects only those rows for which the condition 
# of having any nulls is true.  WE then count the number of rows 
# that survived the filter. 
null_rows_count = df.filter(
    sum(df[col].isNull().cast("int") for col in df.columns) > 0
).count()
print(f"Number of rows with at least one null: {null_rows_count}")

**A:** The number of rows with fields set to null is zero.

**Q:** (from 3.A.6) For numeric data, check for zero.

In [0]:
from pyspark.sql.types import (
    ByteType, ShortType, IntegerType, LongType,
    FloatType, DoubleType, DecimalType
)

numeric_types = (
    ByteType, ShortType, IntegerType, LongType,
    FloatType, DoubleType, DecimalType
)

# printing out to aid interpretation.   Printing the field types is not 
# required by the problem.
for field in df.schema.fields:
    print(f"field={field.name}, type={str(field.dataType)}")

def print_zero_counts(df):
    """
    Print the number of zeroes in each numeric column of the DataFrame.
    """

    # create a list of just the numeric columns.
    numeric_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, numeric_types)
    ]

    # visually confirm that numeric_cols contains just numeric_types.
    print(numeric_cols)

    zero_counts = {}
    for col in numeric_cols:
        count = df.filter(df[col] == 0).count()
        zero_counts[col] = count

    for col, count in zero_counts.items():
        print(f"Number of zeroes in column '{col}': {count}")
        
print_zero_counts(df)


**A:** Although some fields contain values that look numeric, only one column 
(`customer_zip_code_prefix`) was inferred to have a numeric data type by 
`spark.read.csv`. 

However, it is important to distinguish between **data type** and **semantic meaning**.  
The fields `customer_id` and `customer_unique_id` contain hexadecimal-like values 
and are represented as strings. These are identifiers, and we would not perform 
mathematical operations such as addition or subtraction on them. Therefore, they 
are best understood as **categorical (nominal)** variables.

Similarly, although zip code prefixes are stored as numeric values, they do not represent 
quantities. Arithmetic operations on zip code prefixes are meaningless, so they should 
also be treated as **categorical (nominal)** data.

That said, numeric-looking fields can still contain missing data encoded as special 
values. For example, a zip code of `0` could indicate missing data. We checked for 
this possibility and found that there are no entries with a zero zip code.

For identifier fields (`customer_id`, `customer_unique_id`), the presence of a value 
like `'0'` does not necessarily indicate missing data. Determining whether such values 
are invalid would require knowledge of the dataset’s constraints (e.g., whether IDs 
are ever allowed to be zero or repeated). Without such constraints, we cannot conclude 
that these represent missing values.

In [0]:
from pyspark.sql.functions import col

# Check if the string fields contain '0' directly
zero_string_filter = (
    (col("customer_id") == "0") |
    (col("customer_unique_id") == "0")
)
zero_string_count = df.filter(zero_string_filter).count()
print(f"Number of records with string '0' in customer_id or customer_unique_id: {zero_string_count}")

**Q** (from 3.A.6) Are there any fields where zero is reasonable?

**A** I do not think so.  Brazil does not define 0 as a valid zip code.  The `customer_id` and `customer_unique_id` values all appear to be non-zero hexadecimal. None of the other fields are numbers.

**Q** (from 3.A.6) For string categorical data, check for empty strings, "NA", "n/a", and "None" using case-insensitive comparisons. 

In [0]:
from pyspark.sql.functions import lower, trim, col
from functools import reduce
from operator import or_
from pyspark.sql.types import StringType

def print_empty_string_counts(df):
    """
    Print the number of "", "na", "n/a", "none" values in each string column of the DataFrame.
    """
    string_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]
    omit_values = ["", "na", "n/a", "none"]

    for col_name in string_cols:
        count = df.filter(
            lower(trim(col(col_name))).isin(omit_values)
        ).count()
        print(f"Number of '', 'na', 'n/a', 'none' in column '{col_name}': {count}")

def print_rows_with_empty_strings(df):
    """
    Print the number of empty strings in each string column of the DataFrame.
    """

    # create a list of just the numeric columns.
    string_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]
    omit_values = ["", "na", "n/a", "none"]

    omit_filter = reduce(
        or_,
        (lower(trim(col(c))).isin(omit_values) for c in string_cols)
    )

    omit_rows = df.filter(omit_filter)
    display(omit_rows)

print_empty_string_counts(df)

**A**: No fields were found that were empty strings, or that are case insensitive equal to `na`, `n/a`, or `none`.

**Q** (from 3.A.6) Did you find any rows suggesting data has been omitted? If so, how many rows?

**A** I found no evidence of missing data.


**Problem 3.A.7.** Brazil has 26 states and a federal district.  Are there 27 unique state 2-letter codes?

In [0]:
 df.select("customer_state").distinct().count()

**A** There are 27.


**Problem 3.A.8.** Using Spark, compute the number of customers residing in each state (including the federal district) using groupBy. Use Spark to do this. Do not bring all the data to the driver.

Convert the resulting DataFrame to a Pandas DataFrame and create a bar chart using matplotlib.

Why is it safe to bring this DataFrame to the driver without sampling?

In [0]:
# groupby creates a GroupedData object.
# count() agregates the data by the grouping key and returns a new DataFrame with the aggregated values.
# The index of the DataFrame is the grouping key and the values are the aggregated values.
per_state = df.groupby("customer_state").count()
display(per_state)

**Q** (from 3.A.8) Convert the resulting DataFrame to a Pandas DataFrame and create a bar chart using matplotlib.


In [0]:
import matplotlib.pyplot as plt
per_state_pd = per_state.orderBy("customer_state").toPandas()

#display(per_state_pd)

plt.title("Customers per State")
plt.ylabel("number of customers")
plt.xlabel("state or federal district")
plt.bar(per_state_pd["customer_state"], per_state_pd["count"])
plt.show()


**Q** (from Problem 3.A.8) Why is it safe to bring this DataFrame to the driver without sampling?

**A** The number of transacations is substantially larger than the number of states plus the federal district. If we groupBy and then aggregate using count() then the resulting data is counting inside Spark and only the counts are pulled back into the driver.  Since there are only 26 states plus a federal district, the result is now tiny: just 27 numbers.

Let's transform the data to group states into regions.

**Problem 3.A.9.** There are too many states to get an idea of how the customers are 
allocated geographically within Brazil.  Add a column to the DataFrame that denotes region
that is called `region`.
In Brazil states are grouped as follows:

* Southeast Region: Espírito Santo, Minas Gerais, Rio de Janeiro, and São Paulo
* South Region: Paraná, Santa Catarina, and Rio Grande do Sul
* North Region: Acre, Amapá, Amazonas, Pará, Rondônia, Roraima, and Tocantins
* Northeast Region: Alagoas, Bahia, Ceará, Maranhão, Paraíba, Pernambuco, Piauí, Rio Grande do Norte, and Sergipe
* Central-West Region: Goiás, Mato Grosso, Mato Grosso do Sul, and the Federal District.

Are there any customers that do not fall in one of the regions?

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "region",
    when(col("customer_state").isin("SP", "RJ", "ES", "MG"), "Southeast")
    .when(col("customer_state").isin("RS", "SC", "PR"), "South")
    .when(col("customer_state").isin("AC", "AP", "AM", "PA", "RO", "RR", "TO"), "North")
    .when(col("customer_state").isin("AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"), "Northeast")
    .when(col("customer_state").isin("DF", "GO", "MT", "MS"), "Central-West")
    .otherwise(None)
)

**Q** (from 3.A.9) Are there any customers that do not fall in one of the regions?

In [0]:
none_count = df.filter(col("region").isNull()).count()
print(f"Number of rows with region None: {none_count}")

**A** There no customers that do not fall in one of the regions.

**Problem 3.A.10.** Plot a histogram of the number of customers in each region.

This gives us an idea of the scope of the data and what inferences we might be able to draw
regarding regional customer behavior.

In [0]:
import matplotlib.pyplot as plt

per_region = df.groupBy("region").count()
per_region.explain(extended=True)
#display(per_region)

In [0]:
per_region_pd = per_region.orderBy("region").toPandas()
display(per_region_pd)

plt.title("Customers per Region")
plt.ylabel("number of customers")
plt.xlabel("region")
plt.bar(per_region_pd["region"], per_region_pd["count"])
plt.show()



**Problem 3.A.11.** Did adding a region column modify the underlying Spark Dataframe?
Describe lineage and the role of lazy evaluation.

**A** Spark DataFrames are immutable.  Creating a region column created a new DataFrame. 

When we added the `region` column, Spark did not modify the original DataFrame in place. Instead, it created a new DataFrame with an updated logical plan that includes the transformation. Spark uses lazy evaluation, meaning transformations are not executed until an action (like `count()` or `show()`) is called. The lineage of the DataFrame tracks all transformations applied, allowing Spark to recompute results or recover from failures. This approach ensures data integrity and efficient computation.

We did force evaluation when we converted the Spark DataFrame to a Pandas Dataframe.

### Part III.B. Geolocation Data



Now we will perform some EDA and data cleaning on `olist_geolocation_dataset.csv` 

Looking for problems with the geolocation data.

This contains information about the location of zip code prefixes within Brazil.

This data is only about locations of zip code prefixes and not specifically any customers.

**Problem 3.B.1.** Load the `olist_geolocation_dataset.csv` into a 
Spark DataFrame named `geo_df`.  Display the first 10 records.



In [0]:
geo_df = spark.read.csv(os.path.join(DS, "olist_geolocation_dataset.csv"), header=True, inferSchema=True)
display(geo_df.head(10))

# not required by problem:
geo_df.printSchema()

**Problem 3.B.2.** Describe the meaning of the data in each column.  

**Answer 3.B.2**

**geolocation_zip_code_prefix** is the first 5 digits of a Brazilian CEP (postal code). It is stored as an integer, so leading zeros are dropped — for example, the prefix `01046` appears as `1046`.

**geolocation_lat** is the latitude of a location associated with that zip code prefix.

**geolocation_lng** is the longitude of a location associated with that zip code prefix.

**geolocation_city** is the city name associated with that zip code prefix.

**geolocation_state** is the two-letter Brazilian state code, or `DF` for the Federal District.



**Problem 3.B.3.** In the first few rows of the dataset, the zip code prefix `1046` appears more than once. This suggests that the dataset may contain multiple geolocation records for the same zip code prefix.

1. How many rows _are_ in the `geo_df` DataFrame?
2. How many distinct values of `geolocation_zip_code_prefix`` are present?

What does this tell you about the structure of the dataset?

**Follow-up.** Why might a dataset contain multiple latitude/longitude entries for the same zip code prefix?

You may search for external information, but also provide at least one plausible explanation based on your understanding of how such data might be collected.


In [0]:
row_count = geo_df.count()
distinct_zip_count = geo_df.select("geolocation_zip_code_prefix").distinct().count()
print(f"Number of rows: {row_count}")
print(f"Number of distinct zip code prefixes: {distinct_zip_count}")

In [0]:
from pyspark.sql.functions import length

count_more_than_4_digits = geo_df.filter(col("geolocation_zip_code_prefix") > 9999).select("geolocation_zip_code_prefix").distinct().count()
print(f"'geolocation_zip_code_prefix` entries wtih more than 4 digits: {count_more_than_4_digits}")
count_more_than_5_digits = geo_df.filter(col("geolocation_zip_code_prefix") > 99999).select("geolocation_zip_code_prefix").distinct().count()
print(f"`geolocation_zip_code_prefix` entries wtih more than 5 digits: {count_more_than_5_digits}")


Brazilian zip codes (CEPs) take the form `NNNNN-NNN`, e.g., `12345-321`. 
The dataset stores only the first 5 digits as `geolocation_zip_code_prefix`.

Some prefixes appear to have only 4 digits (e.g., `1046`), but this is because 
they are stored as integers and leading zeros are dropped. For example, `1046` 
likely represents the prefix `01046`.

It is possible that the original data contained full CEPs, and multiple CEPs 
sharing the same first 5 digits are represented as separate rows after 
truncation, resulting in multiple geolocation records per prefix.

**Problem  3.B.4.** Write code to determine if there is any missing data in the `geo_df` DataFrame.
Run the code on the DataFrame.  Is there any missing data?

In [0]:

print_zero_counts(geo_df)
print_empty_string_counts(geo_df)

No fields that appear to be missing data.

**Problem 3.B.5.** In subsequent problems we will use `matplotlib` to visualize the data, but `matplotlib` operates on data that resides in the memory of the Python interpreter running on the driver node. Calling `toPandas()` on a Spark DataFrame collects all data to the driver, which may cause an Out-of-Memory (OOM) error if the dataset is too large.

To determine whether this is safe, we will estimate the memory required by the full dataset.

1. Use Spark’s `sample` method to select a small fraction of the dataset.
2. Convert the sample to a Pandas DataFrame using `toPandas()`.
3. Use `memory_usage(deep=True)` to compute the memory usage of the sample.
4. Based on the sample fraction, estimate the memory required for the full dataset.
5. Inspect the memory of the driver node using one of the following (see *Inspecting Driver Memory* below)
6. Compare your estimated memory requirement to the **available memory** on the driver.

**Question:** Based on your estimate, would it be safe to call `toPandas()` on the full dataset? Justify your answer.

**Question:** What assumptions are you making when extrapolating from the sample? Under what conditions might this estimate be inaccurate?

#### Inspecting Driver Memory

There are several ways to inspect the memory in the driver node including.

```
  psutil.virtual_memory()
```

and

```
  pages = os.sysconf("SC_PHYS_PAGES")
  page_size = os.sysconf("SC_PAGE_SIZE")
  total_mem = pages * page_size
```

The function `psutil.virtual_memory()` is poorly named because it returns an object with many fields providing information about the memory of the driver node.  Which fields are present depends on the operating system.   See https://psutil.readthedocs.io/latest/api.html#memory

ASIDE: I found there was more than enough memory in the driver, which makes it easier to 
visualize the raw data without having to employ special considerations for outliers.


**Answer 3.B.5**

In [0]:
import psutil
import os

print(f"rows in geo_df: {geo_df.count()}")

def extrapolate_pd_size_from_sample(df):

    sample_fraction = 0.01
    sample_pd = df.sample(False, sample_fraction, seed=42).toPandas()
    sample_mem = sample_pd.memory_usage(deep=True).sum()

    estimated_full_mem = sample_mem / sample_fraction
    return estimated_full_mem

print(f"Estimated full pandas memory: {extrapolate_pd_size_from_sample(df) / (1024**3):.2f} GB")

 # aside: `virtual_memory` this is a poorly named fucntion because it returns more than just VM
 # the size of virtual memory.
mem = psutil.virtual_memory()
print(f"Total memory: {mem.total / (1024**3):.2f} GB")
print(f"Available memory: {mem.available / (1024**3):.2f} GB")


pages = os.sysconf("SC_PHYS_PAGES")
page_size = os.sysconf("SC_PAGE_SIZE")
total_mem = pages * page_size

print(f"Total memory (SC_PHYS_PAGES * SC_PAGE_SIZE): {total_mem / (1024**3):.2f} GB")

## spark.driver.memory is not available to the free edition of Databricks.
#spark_conf = spark.conf.getAll(); spark_mem = spark_conf.get("spark.driver.memory", "Not set")
#print(f"Spark config: spark.driverr.memory: {spark_mem}")


The estimated size of the dataset appears small enough that it is unlikely to cause memory issues for the driver.

However, this estimate may be inaccurate if there is high variance in the size of individual rows and the sample is not representative. In particular, if the sample fails to include large or complex rows (e.g., rows with long strings), we may underestimate the true memory requirements. 

Additionally, the estimate assumes that memory usage scales linearly with the number of rows, which may not hold exactly due to overhead in Pandas data structures or differences in data types.

**Problem 3.B.6.** Plot the latitude and longitude of the zip code prefixes on a scatter plot and
overlay a bounding box for the min and max latitude and longitude of Brazil with a 1 degree
of margin around the westernmost, easternmost, northernmost, and southernmost bounds. 
You may use `toPandas()` on the DataFrame so you can plot the data, but this brings the 
data into the driver.  Aside: In the previous problem, we explored whether loading all the
data into memory is likely to OOM the driver, and found that it is not.



In [0]:
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle

def dms(degrees, minutes, seconds, direction):
    decimal = degrees + minutes/60 + seconds/3600
    if direction in ['S', 'W']:
        decimal = -decimal
    return decimal

w_long = dms(73, 59, 32, "W") - 1
e_long = dms(32, 24, 35, "W") + 1
n_lat = dms(5, 0, 0, "N") + 1
s_lat = dms(33, 0, 0, "S") - 1

geo_pd = geo_df.select("geolocation_lng", "geolocation_lat").toPandas()
plt.figure(figsize=(10, 4))
plt.title("distribution of zip code prefixes")
plt.xlabel("longitude")
plt.ylabel("latitude")

width = e_long - w_long
height = n_lat - s_lat

bbox = Rectangle((w_long, s_lat), width, height, linewidth=2, edgecolor='orange', facecolor='none')
plt.gca().add_patch(bbox)

plt.scatter(geo_pd["geolocation_lng"], geo_pd["geolocation_lat"], s=2)
plt.show()

**Problem 3.B.7.** Do any zip code prefixess fall outside the margin surrounding the bounding box
containing of Brazil?  If so, write and execute code to count how many fall outside the 
margin and execute.

Use Spark to determine how many distinct zip code prefixes have no geo locations that are inside the bounding box of Brazil, and how many have both geo locations inside and outside of the margin bounding box of Brazil.

What could this mean?  How could this happen? This is just conjecture, but it is worth thinking about because such things are not uncommon in real data.

In [0]:
from pyspark.sql.functions import col

n_outside = geo_df.filter(
    (col("geolocation_lng") < w_long) |
    (col("geolocation_lng") > e_long) |
    (col("geolocation_lat") > n_lat) |
    (col("geolocation_lat") < s_lat)
).count()
display(n_outside)


31 zip code prefixes fall outside the bounds of Brazil!

In [0]:
from pyspark.sql.functions import col, when, countDistinct
from pyspark.sql.functions import sum as spark_sum

# Add column specifying inside/outside Brazil margin bounding box
geo_df_flagged = geo_df.withColumn(
    "inside_brazil",
    when(
        (col("geolocation_lng") >= w_long) &
        (col("geolocation_lng") <= e_long) &
        (col("geolocation_lat") >= s_lat) &
        (col("geolocation_lat") <= n_lat),
        1
    ).otherwise(0)
)

# Group by zip code prefix and count how many have all location outside Brazil.
zip_prefixes_all_out = (
    geo_df_flagged
    .groupBy("geolocation_zip_code_prefix")
    .agg(spark_sum("inside_brazil").alias("num_inside_brazil"))
    .filter(col("num_inside_brazil") == 0)
)
print(f"Number of zip code prefixes wherein all lat-longs are outside Brazil: {zip_prefixes_all_out.count()}")

# Group by zip code prefix and count how many have both inside and outside locations
zip_prefixes_both = geo_df_flagged.groupBy("geolocation_zip_code_prefix") \
    .agg(countDistinct("inside_brazil").alias("inside_outside_count")) \
    .filter(col("inside_outside_count") > 1) \
    .count()

print(f"Number of zip code prefixes with some inside and som eoutside Brail: {zip_prefixes_both}")


16 zip code prefixes have at least one lat-long coordinate that is inside
while the other is outside.  That still leaves 4 that are outside the bounds of Brazil with
no lat-long within.

Having such disparate lat-longs could occur with data entry errors or instrumentation errors.
These are not likely legitimate.  We might be able to reasonably clean/repair the 16.

**Problem 3.B.8.** Group all rows by city-state pair.  How many cities are
represented in the data? 

In [0]:
city_state_count = geo_df.select("geolocation_city", "geolocation_state").distinct().count()
display(city_state_count)

There are 8463 city-state pairs.

**Problem 3.B.9.** Let's evaluate whether using a robust estimate of centrality eliminates invalid outliers.

1. Using Spark, group `geo_df` by `geolocation_zip_code_prefix` and compute the median latitude and median longitude for each group.  Assign the resulting dataframe to `geo_df_median`.
2. Use `toPandas()` to bring the aggregated DataFrame to the driver, and create a scatter plot using `matplotlib` of the resulting coordinates. Overlay the bounding box for Brazil as in the earlier plot.
3. Print any rows in `geo_df` that have zip code prefixes whose median latitude/longitude still fall outside the bounding box of Brazil.
4. From the rows printed out in 3, find the two cities with the most zip code prefixes.  Use Google Maps to lookup the latitude and longitude of those cities.  What does this suggest about the data?
5. Identify how many from Step 3 correspond to single rows.
5. Suggest a strategy to handle these remaining outliers. You don't need to implement _it_.

In [0]:
from pyspark.sql.functions import expr

# Aggregate to median latitude and longitude per zip code prefix
geo_df_median = geo_df.groupBy("geolocation_zip_code_prefix") \
    .agg(
        expr("percentile_approx(geolocation_lat, 0.5)").alias("median_lat"),
        expr("percentile_approx(geolocation_lng, 0.5)").alias("median_lng")
    )

# Convert to Pandas for plotting
geo_pd_median = geo_df_median.toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.title("Median lat-long per zip code prefix")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.scatter(geo_pd_median["median_lng"], geo_pd_median["median_lat"], s=2)

width = e_long - w_long
height = n_lat - s_lat

bbox = Rectangle((w_long, s_lat), width, height, linewidth=2, edgecolor='orange', facecolor='none')
plt.gca().add_patch(bbox)

plt.show()

outside_df = geo_df_median.filter(
    (col("median_lng") < w_long) |
    (col("median_lng") > e_long) |
    (col("median_lat") > n_lat) |
    (col("median_lat") < s_lat)
)
n_outside = outside_df.count()
print(f"Number of median lat-longs that remain outside the bounding box of Brazil: {n_outside}")

outside_zip_prefixes = outside_df.select("geolocation_zip_code_prefix")
geo_df_outside = geo_df.join(
    outside_zip_prefixes,
    on="geolocation_zip_code_prefix",
    how="inner"
).orderBy("geolocation_zip_code_prefix")
display(geo_df_outside)

group_counts = geo_df_outside.groupBy("geolocation_zip_code_prefix").count()
display(group_counts)

single_row_prefixes = group_counts.filter(col("count") == 1)
n_single_row = single_row_prefixes.count()
print(f"Number of zip code prefixes outside Brazil with only one row: {n_single_row}")
#geo_df_outside.groupBy("geolocation_zip_code_prefix").agg()



Zip-code prefix 68275 has the most rows fitlered from `geo_df`.  All but one of its rows refers to Porto Trombetas, PA. According to Google Maps, Proto Tromboles is at 1.4595° S latitude and 56.3745° W longitude.  None of the latitude or longitude values are very close.  I pasted all of row coordinates in Google Maps and found that one lands in the area around Porto, Portugal rather than Porto Trombeta, PA, Brazil.  Two row coordinates land in Puebla de Sanabria, Spain. 

The repetition of with Puebla de Sanabria, Spain suggests a systematic error rather than random noise. The Porto, Portugal entries may be due to ambiguity in the name "Porto" or an autocomplete issue during geocoding. The Spain entries are less obvious, but also indicate incorrect geocoding.

Zip-code prefix 29654 has the second most rows filtered from `geo_df`.  All of its rows refer to Santo Antonio do Canaa, ES.
⁠According to Google Maps, Santo Antônio do Canaã, located in the state of Espírito Santo, Brazil, are approximately 19.8251° S latitude and 40.6574° W longitude.  Two of its rows agree reasonably closely with Google Maps, but two do not.   The two that do not are also not close to each other.   The two that are not close to other are at San Antonio, TX, USA and Ocampo, Mexico respectively. 

These examples suggest that the dataset contains a mix of correct coordinates and significant geocoding errors, likely due to ambiguous place names or incorrect matches during data collection.

There are three singleton cases (i.e., zip-code prefixes that appear only once) whose latitude and longitude fall outside Brazil. Without additional context or corroborating data, these cannot be corrected reliably.

One reasonable solution for such clearly erroneous data is to replace the values for their location with the latitude and longitdue from Google Maps for their respective cities.   When there are small number of datapoints involved as in this case we may be able to simply drop the row.  It depends on what we are using the data for.



**Problem 3.B.10.** Assuming we replacing the obviously erroneous with the median latitude
and latitude, let's see how much using the median latitude and latitude for all zip code
prefixes would affect substantilly move zip codes prefixes `geolocation_lat` `geolocation_lng`
coordinates within the Brazil.  

1. Assign `geo_df_median` to a new DataFrame that removes all zip codes from `geo_df_median` 
   that have a median latitude and median longitude that is
   outside Brazil.  We are assuming this is okay for our purposes.  Fortunately
   we do not delete the original data so we could always go back and change this assumption
   for applications that require all the rows even with invalid geo locations.
2. Derive a new DataFrame `geo_df_both` from the `geo_df` with added columns labelled `median_lat` and 
   `median_lng`.  `median_lat` and `median_lng` come by joining with `geo_df_median`.
3. Consideing only zip code prefixes wihtin the bounding box of Brazil, 
   create a scatter plot that shows all `median_lat` and `median_lng` and all
   `geolocation_lat` and `geolocation_lng` that are more than 200km from the associated 
   `median_lat` and `median_lng`.
4. Print the number and fraction of zip code prefixes in `geo_df_both` that have
   a `geolocation` and `geolocation_lng` coordinate that is more than 200km from the associated
   `median_lat` and `median_lng`.

What conclusions would you draw from the number of zip code rows that would be moved significantly
by using media latitude and longitude?


In [0]:
import math

def geodesic_miles(lat1, lon1, lat2, lon2):
    """
    Compute the great-circle distance between two points on Earth using
    the haversine formula.

    Parameters
    ----------
    lat1, lon1 : float
        Latitude and longitude of the first point in degrees.

    lat2, lon2 : float
        Latitude and longitude of the second point in degrees.

    Returns
    -------
    float
        Distance in miles along the Earth's surface.

    """

    # Mean Earth radius in miles
    R = 3958.7613
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# If the median lat-long is outside Brazil then for this data pipeline
# we assume that row is not usable and discard it.
geo_df_median = geo_df_median.filter(
    (col("median_lng") >= w_long) &
    (col("median_lng") <= e_long) &
    (col("median_lat") >= s_lat) &
    (col("median_lat") <= n_lat)
)

# geo_df_both has both geolocation_lat, geolocation_lng and median_lat, median_lng coordinates.
geo_df_both = geo_df.join(
    geo_df_median,
    on="geolocation_zip_code_prefix",
    how="inner"  # Keep only rows whose zip prefix appears in geo_df_median.
)

display(geo_df_both.limit(5))
import matplotlib.pyplot as plt
geo_df_both = geo_df_both.filter(
    (col("geolocation_lng") >= w_long) &
    (col("geolocation_lng") <= e_long) &
    (col("geolocation_lat") >= s_lat) &
    (col("geolocation_lat") <= n_lat)
)
# Convert Spark DataFrame to Pandas for plotting and distance calculation
geo_pd_both = geo_df_both.toPandas()

# Calculate distance in km between each point and its median
geo_pd_both["distance_km"] = geo_pd_both.apply(
    lambda row: geodesic_miles(
        row["geolocation_lat"], row["geolocation_lng"],
        row["median_lat"], row["median_lng"]
    ) / 0.6,
    axis=1
)

# Points more than 200km from their median
far_points = geo_pd_both[geo_pd_both["distance_km"] > 200]

plt.figure(figsize=(10, 6))
plt.scatter(geo_pd_both["median_lng"], geo_pd_both["median_lat"], s=10, c="blue", label="Median Coordinates")
plt.scatter(far_points["geolocation_lng"], far_points["geolocation_lat"], s=10, c="red", label=">200km from Median")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Median Coordinates and Points >200km from Median")
plt.legend()
plt.show()

num_zip_prefixes_far = far_points["geolocation_zip_code_prefix"].nunique()
print(f"Number of zip code prefixes with points >200km from their median: {num_zip_prefixes_far}")
num_zip_prefixes_total = geo_pd_both["geolocation_zip_code_prefix"].nunique()
fraction_zip_prefixes_far = num_zip_prefixes_far / num_zip_prefixes_total
print(f"Total zip code prefixes: {num_zip_prefixes_total}")
print(f"Fraction of zip code prefixes with points >200km from their median: {fraction_zip_prefixes_far:.4f}")

Most zip code prefixes' geo locations do not move long distances when using the median latitude and longitude of the rows that share the same zip code prefix. However, the fact that we have this much noise to begin with means we may want to use Google Maps or a similar data source to first replace all zip code rows with the city latitude and longitude if they are more than a reasonable distance (e.g., 100km or 200km) from the city. It depends on how we intend to use the data.

### COMMENT ON EDA AND CLEANING

Normally we would continue with this process of EDA and data cleaning for every dataset 
appearing in the problem.  For purposes of keeping the homework from becoming
exorbitantly lengthy, we will stop the cleaning and EDA here and move on to hypothesis tests.

## Part IV. Hypothesis Tests

I want to perform at least one hypothesis test on the data.

**Problem 4.1:** Test for differences in purchase amounts between Rio de Janeiro and 
Sao Paolo.  \\(H_A\\): "The average purchase amount per order in São Paulo is different from that in Rio de Janeiro."  Use a t-test.

In [0]:
from pyspark.sql import functions as F
from scipy import stats

# Load orders and payments
orders_df = spark.read.csv(
    os.path.join(DS, "olist_orders_dataset.csv"),
    header=True, inferSchema=True
)
payments_df = spark.read.csv(
    os.path.join(DS, "olist_order_payments_dataset.csv"),
    header=True, inferSchema=True
)

# Sum payment_value per order (an order can have multiple payment installments)
order_totals = payments_df.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment")
)

# Join: orders → customers → payment totals
order_state = (
    orders_df
    .join(df.select("customer_id", "customer_state"), on="customer_id", how="inner")
    .join(order_totals, on="order_id", how="inner")
)

# Filter for SP and RJ, collect to driver
sp_payments = (
    order_state
    .filter(col("customer_state") == "SP")
    .select("total_payment")
    .toPandas()["total_payment"]
    .dropna()
    .values
)
rj_payments = (
    order_state
    .filter(col("customer_state") == "RJ")
    .select("total_payment")
    .toPandas()["total_payment"]
    .dropna()
    .values
)

print(f"SP orders: {len(sp_payments)}, mean = {sp_payments.mean():.2f}")
print(f"RJ orders: {len(rj_payments)}, mean = {rj_payments.mean():.2f}")
print(f"Observed difference (SP - RJ): {sp_payments.mean() - rj_payments.mean():.4f}")

t_stat, p_value = stats.ttest_ind(sp_payments, rj_payments, equal_var=False)
print(f"\nWelch's t-test: t = {t_stat:.4f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Result: reject H_0 at α=0.05 — evidence of a difference in mean purchase amount.")
else:
    print("Result: fail to reject H_0 at α=0.05 — no significant difference detected.")

**Problem 4.2:** Repeat problem 4.1 using a permutation test.  Plot a histogram of the permutation differences
betwen Sao Paolo and Rio de Janeiro.  Place a vertical line at the observed difference between
Rio de Janeiro and Sao Paolo based on the original data.  Compute the p-value.  Does the result deviate from the t-test?

In [0]:
import numpy as np
import matplotlib.pyplot as plt

# sp_payments and rj_payments carried over from Problem 4.1.
observed_diff = sp_payments.mean() - rj_payments.mean()

rng = np.random.default_rng(seed=42)
combined = np.concatenate([sp_payments, rj_payments])
n_sp = len(sp_payments)
n_permutations = 10_000

perm_diffs = np.empty(n_permutations)
for i in range(n_permutations):
    shuffled = rng.permutation(combined)
    perm_diffs[i] = shuffled[:n_sp].mean() - shuffled[n_sp:].mean()

p_value_perm = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

plt.figure(figsize=(8, 4))
plt.hist(perm_diffs, bins=60, edgecolor="none", alpha=0.75, label="Permutation differences")
plt.axvline(observed_diff, color="red", linewidth=2, label=f"Observed diff = {observed_diff:.2f}")
plt.axvline(-observed_diff, color="red", linewidth=2, linestyle="--")
plt.xlabel("Mean(SP) − Mean(RJ)")
plt.ylabel("Count")
plt.title("Permutation Test: SP vs RJ Purchase Amount")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Observed difference (SP - RJ): {observed_diff:.4f}")
print(f"Permutation p-value (two-tailed): {p_value_perm:.4f}")
if p_value_perm < 0.05:
    print("Result: reject H_0 at α=0.05.")
else:
    print("Result: fail to reject H_0 at α=0.05.")

**Problem 4.3:** Test for Association Between Review Scores and Delivery Time. Null Hypothesis \\(H_0\\): There is no correlation between delivery time and review scores.
Alternative Hypothesis \\(H_A\\): There is a negative correlation between delivery time and review scores, indicating that longer delivery times correlate with lower review scores.  Use Pearson correlation.

In [0]:
from pyspark.sql import functions as F
from scipy import stats
import sys

# reviews has multi-line reviews.
reviews_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(os.path.join(DS, "olist_order_reviews_dataset.csv"))
)

# Load orders
orders_df = spark.read.csv(
    os.path.join(DS, "olist_orders_dataset.csv"),
    header=True, inferSchema=True
)

# Compute delivery time in days for delivered orders only.
delivery_df = (
    orders_df
    .filter(col("order_status") == "delivered")
    .filter(col("order_delivered_customer_date").isNotNull())
    .withColumn(
        "delivery_days",
        (F.unix_timestamp("order_delivered_customer_date") -
         F.unix_timestamp("order_purchase_timestamp")) / 86400.0
    )
    .select("order_id", "delivery_days")
)

#display(delivery_df.limit(10))

# Average review score per order (a small number of orders have multiple reviews).
review_agg = reviews_df.groupBy("order_id").agg(
    F.avg("review_score").alias("review_score")
)

#display(review_agg.limit(10))

result_pd = (
    delivery_df
    .join(review_agg, on="order_id", how="inner")
    .dropna()
    .toPandas()
)

delivery_days  = result_pd["delivery_days"].values
review_scores  = result_pd["review_score"].values

print(f"Sample size: {len(delivery_days)}")
print(f"Mean delivery days: {delivery_days.mean():.2f}")
print(f"Mean review score:  {review_scores.mean():.2f}")

r, p_value = stats.pearsonr(delivery_days, review_scores)
print(f"\nPearson r = {r:.4f},  p = {p_value:.2e}")
if p_value < 0.05 and r < 0:
    print("Result: reject H_0 — significant negative correlation between delivery time and review score.")
elif p_value < 0.05:
    print("Result: reject H_0 — significant correlation detected (but positive, not negative).")
else:
    print("Result: fail to reject H_0 at α=0.05 — no significant correlation detected.")

**Problem 4.4:** Repeat 4.3 using a permutation test.  Plot a histogram of the distribution of
Pearson correlations for each permutation and place a vertical line at the measured correlation 
for the original correlation between review score and delivery time.
Compute the p-value.  Does the result of 4.4 deviate from the result in 4.3?

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# delivery_days and review_scores carried over from Problem 4.3.
observed_r, _ = stats.pearsonr(delivery_days, review_scores)

rng = np.random.default_rng(seed=42)
n_permutations = 1000

perm_r = np.empty(n_permutations)
for i in range(n_permutations):
    shuffled_scores = rng.permutation(review_scores)
    perm_r[i], _ = stats.pearsonr(delivery_days, shuffled_scores)

# One-tailed p-value: H_A is r < 0, so count fraction of permutations with r <= observed_r.
p_value_perm = np.mean(perm_r <= observed_r)

plt.figure(figsize=(8, 4))
plt.hist(perm_r, bins=60, edgecolor="none", alpha=0.75, label="Permutation r values")
plt.axvline(observed_r, color="red", linewidth=2, label=f"Observed r = {observed_r:.4f}")
plt.xlabel("Pearson r")
plt.ylabel("Count")
plt.title("Permutation Test: Delivery Time vs Review Score")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Observed Pearson r = {observed_r:.4f}")
print(f"Permutation p-value (one-tailed, H_A: r < 0): {p_value_perm:.4f}")
if p_value_perm < 0.05:
    print("Result: reject H_0 at α=0.05 — permutation test confirms negative correlation.")
else:
    print("Result: fail to reject H_0 at α=0.05.")